# Analysis of American Parlimentary Debate Association Rounds Fall 2018 - Spring 2026

## Introduction

...

## Data Curation

The data for this project was sourced at the APDA online forum where, after the conclusion of each tournament, results are posted in the form of PDF Tab Cards. Tab Cards are structured by team where each team has its own table within the PDF. Each table is labelled with the team name, and contains rows that detail the round number, weather the team was in Government or Opposition position, win/loss status, the opponent team name, judge name, the speaks and ranks for each team member, and the total speaks and ranks of the team. For the preliminary data curation, I used pdfplumber to split each PDF into tables in order to extract data for each team. Bounding boxes were used to extract the team names that prefaced each table. Additionally, a reference was necesary to match the team names, which differ by tournament, to the individual debaters in order to creat the columns for the opponent names. It was also necesary to collapse the mirror rows as each match is represented twice, one for the Government teams table and the other in the Opposition team table. Regex patterns were used for cleaning and normalizing the data.  

In [795]:
# imports used for parsing data from PDFs
import re
import logging
import pandas as pd
import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [796]:
# below contains all the PDFs for the tournaments Spring2019 - Spring2026
SPRING_2026_TAB_CARDS = [
    ("Spring2026/American_Spring2026_Tab_Card.pdf", "American", "Spring", 2026), 
    ("Spring2026/Amherst_Spring2026_Tab_Card.pdf", "Amherst", "Spring", 2026), 
    ("Spring2026/Boston_Spring2026_Tab_Card.pdf", "Boston", "Spring", 2026),
    ("Spring2026/Brandeis_Spring2026_Tab_Card.pdf", "Brandeis", "Spring", 2026),
    ("Spring2026/Brown_Spring2026_Tab_Card.pdf", "Brown", "Spring", 2026),
    ("Spring2026/CMU_Spring2026_Tab_Card.pdf", "CMU", "Spring", 2026),
    ("Spring2026/Columbia_Spring2026_Tab_Card.pdf", "Columbia", "Spring", 2026),
    ("Spring2026/Delaware_Spring2026_Tab_Card.pdf", "Delaware", "Spring", 2026),
    ("Spring2026/Drexel_Spring2026_Tab_Card.pdf", "Drexel", "Spring", 2026),
    ("Spring2026/GU_Spring2026_Tab_Card.pdf", "Georgetown", "Spring", 2026),
    ("Spring2026/GW_Spring2026_Tab_Card.pdf", "GW", "Spring", 2026),
    ("Spring2026/Northeastern_Spring2026_Tab_Card.pdf", "Northeastern", "Spring", 2026),
    ("Spring2026/Penn_Spring2026_Tab_Card.pdf", "Penn", "Spring", 2026),
    ("Spring2026/Princeton_Spring2026_Tab_Card.pdf", "Princeton", "Spring", 2026),
    ("Spring2026/Rutgers_Spring2026_Tab_Card.pdf", "Rutgers", "Spring", 2026), 
    ("Spring2026/Swarthmore_Spring2026_Tab_Card.pdf", "Swarthmore", "Spring", 2026), 
    ("Spring2026/UMass_Amherst_Spring2026_Tab_Card.pdf", "UMass Amherst", "Spring", 2026), 
    ("Spring2026/UMD_Spring2026_Tab_Card.pdf", "UMD", "Spring", 2026), 
    ("Spring2026/UVA_Spring2026_Tab_Card.pdf", "UVA", "Spring", 2026), 
    ("Spring2026/Yale_Spring2026_Tab_Card.pdf", "Yale", "Spring", 2026) 
]

FALL_2025_TAB_CARDS = [
    ("Fall2025/Binghamton_Fall2025_Tab_Card.pdf", "Binghamton", "Fall", 2025),
    ("Fall2025/Brown_Fall2025_Tab_Card.pdf", "Brown", "Fall", 2025),  
    ("Fall2025/Boston_Fall2025_Tab_Card.pdf", "Boston", "Fall", 2025),   
    ("Fall2025/CMU_Fall2025_Tab_Card.pdf", "CMU", "Fall", 2025),   
    ("Fall2025/Fordham_Fall2025_Tab_Card.pdf", "Fordham", "Fall", 2025),
    ("Fall2025/GW_Fall2025_Tab_Card.pdf", "GW", "Fall", 2025),   
    ("Fall2025/Harvard_Fall2025_Tab_Card.pdf", "Harvard", "Fall", 2025),
    ("Fall2025/Hopkins_Fall2025_Tab_Card.pdf", "John's Hopkins", "Fall", 2025),
    ("Fall2025/Northeastern_Fall2025_Tab_Card.pdf", "Northeastern", "Fall", 2025),
    ("Fall2025/Pitt_Fall2025_Tab_Card.pdf", "Pitt", "Fall", 2025),
    ("Fall2025/Swarthmore_Fall2025_Tab_Card.pdf", "Swarthmore", "Fall", 2025),
    ("Fall2025/Temple_Fall2025_Tab_Card.pdf", "Temple", "Fall", 2025),
    ("Fall2025/Tufts_Fall2025_Tab_Card.pdf", "Tufts", "Fall", 2025),
    ("Fall2025/UMD_Fall2025_Tab_Card.pdf", "UMD", "Fall", 2025),
    ("Fall2025/UVA_Fall2025_Tab_Card.pdf", "UVA", "Fall", 2025),
    ("Fall2025/Wellesley_Fall2025_Tab_Card.pdf", "Wellesley", "Fall", 2025),
    ("Fall2025/Wesleyan_Fall2025_Tab_Card.pdf", "Wesleyan", "Fall", 2025),
    ("Fall2025/William&Mary_Fall2025_Tab_Card.pdf", "William & Mary", "Fall", 2025),
    ("Fall2025/Williams_Fall2025_Tab_Card.pdf", "Williams", "Fall", 2025),
]

SPRING_2025_TAB_CARDS = [
    ("Spring2025/American_Spring2025_Tab_Card.pdf", "American", "Spring", 2025),
    ("Spring2025/Boston_Spring2025_Tab_Card.pdf", "Boston", "Spring", 2025),
    ("Spring2025/Brandeis_Spring2025_Tab_Card.pdf", "Brandeis", "Spring", 2025),
    ("Spring2025/Brown_Spring2025_Tab_Card.pdf", "Brown", "Spring", 2025),
    ("Spring2025/Chicago_Spring2025_Tab_Card.pdf", "Chicago", "Spring", 2025),
    ("Spring2025/CMU_Spring2025_Tab_Card.pdf", "CMU", "Spring", 2025),
    ("Spring2025/Drexel_Spring2025_Tab_Card.pdf", "Drexel", "Spring", 2025),
    ("Spring2025/Georgetown_Spring2025_Tab_Card.pdf", "Georgetown", "Spring", 2025),
    ("Spring2025/GW_Spring2025_Tab_Card.pdf", "GW", "Spring", 2025),
    ("Spring2025/Hopkins_Spring2025_Tab_Card.pdf", "John's Hopkins", "Spring", 2025),
    ("Spring2025/Northeastern_Spring2025_Tab_Card.pdf", "Northeastern", "Spring", 2025),
    ("Spring2025/Princeton_Spring2025_Tab_Card.pdf", "Princeton", "Spring", 2025),
    ("Spring2025/Swarthmore_Spring2025_Tab_Card.pdf", "Swarthmore", "Spring", 2025),
    ("Spring2025/Temple_Spring2025_Tab_Card.pdf", "Temple", "Spring", 2025),
    ("Spring2025/Tufts_Spring2025_Tab_Card.pdf", "Tufts", "Spring", 2025),
    ("Spring2025/UMass_Amherst_Spring2025_Tab_Card.pdf", "UMass Amherst", "Spring", 2025),
    ("Spring2025/UMD_Spring2025_Tab_Card.pdf", "UMD", "Spring", 2025),
    ("Spring2025/UVA_Spring2025_Tab_Card.pdf", "UVA", "Spring", 2025),
    ("Spring2025/Wellesley_Spring2025_Tab_Card.pdf", "Wellesley", "Spring", 2025),
    ("Spring2025/William&Mary_Spring2025_Tab_Card.pdf", "William & Mary", "Spring", 2025),
    ("Spring2025/Yale_Spring2025_Tab_Card.pdf", "Yale", "Spring", 2025),
]

FALL_2024_TAB_CARDS = [
    ("Fall2024/American_Fall2024_Tab_Card.pdf", "American", "Fall", 2024),
    ("Fall2024/Bentley_Fall2024_Tab_Card.pdf", "Bentley", "Fall", 2024),
    ("Fall2024/Brandeis_Fall2024_Tab_Card.pdf", "Brandeis", "Fall", 2024),
    ("Fall2024/Brown_Fall2024_Tab_Card.pdf", "Brown", "Fall", 2024),
    ("Fall2024/CMU_Fall2024_Tab_Card.pdf", "CMU", "Fall", 2024),
    ("Fall2024/Columbia_Fall2024_Tab_Card.pdf", "Columbia", "Fall", 2024),
    ("Fall2024/Drexel_Fall2024_Tab_Card.pdf", "Drexel", "Fall", 2024),
    ("Fall2024/Fordham_Fall2024_Tab_Card.pdf", "Fordham", "Fall", 2024),
    ("Fall2024/Georgetown_Fall2024_Tab_Card.pdf", "Georgetown", "Fall", 2024),
    ("Fall2024/GW_Fall2024_Tab_Card.pdf", "GW", "Fall", 2024),
    ("Fall2024/Hopkins_Fall2024_Tab_Card.pdf", "John's Hopkins", "Fall", 2024),
    ("Fall2024/Northeastern_Fall2024_Tab_Card.pdf", "Northeastern", "Fall", 2024),
    ("Fall2024/NYU_Fall2024_Tab_Card.pdf", "NYU", "Fall", 2024),
    ("Fall2024/Pitt_Fall2024_Tab_Card.pdf", "Pitt", "Fall", 2024),
    ("Fall2024/Swarthmore_Fall2024_Tab_Card.pdf", "Swarthmore", "Fall", 2024),
    ("Fall2024/UMD_Fall2024_Tab_Card.pdf", "UMD", "Fall", 2024),
    ("Fall2024/UVA_Fall2024_Tab_Card.pdf", "UVA", "Fall", 2024),
    ("Fall2024/Wesleyan_Fall2024_Tab_Card.pdf", "Wesleyan", "Fall", 2024),
]

SPRING_2024_TAB_CARDS = [
    ("Spring2024/American_Spring2024_Tab_Card.pdf", "American", "Spring", 2024),
    ("Spring2024/Boston_Spring2024_Tab_Card.pdf", "Boston", "Spring", 2024),
    ("Spring2024/Brown_Spring2024_Tab_Card.pdf", "Brown", "Spring", 2024),
    ("Spring2024/CMU_Spring2024_Tab_Card.pdf", "CMU", "Spring", 2024),
    ("Spring2024/Georgetown_Spring2024_Tab_Card.pdf", "Georgetown", "Spring", 2024),
    ("Spring2024/Hopkins_Spring2024_Tab_Card.pdf", "John's Hopkins", "Spring", 2024),
    ("Spring2024/Northeastern_Spring2024_Tab_Card.pdf", "Northeastern", "Spring", 2024),
    ("Spring2024/NYU_Spring2024_Tab_Card.pdf", "NYU", "Spring", 2024),
    ("Spring2024/Penn_Spring2024_Tab_Card.pdf", "Penn", "Spring", 2024),
    ("Spring2024/Pitt_Spring2024_Tab_Card.pdf", "Pitt", "Spring", 2024),
    ("Spring2024/Princeton_Spring2024_Tab_Card.pdf", "Princeton", "Spring", 2024),
    ("Spring2024/Rutgers_Spring2024_Tab_Card.pdf", "Rutgers", "Spring", 2024),
    ("Spring2024/Swarthmore_Spring2024_Tab_Card.pdf", "Swarthmore", "Spring", 2024),
    ("Spring2024/Temple_Spring2024_Tab_Card.pdf", "Temple", "Spring", 2024),
    ("Spring2024/UMD_Spring2024_Tab_Card.pdf", "UMD", "Spring", 2024),
    ("Spring2024/UVA_Spring2024_Tab_Card.pdf", "UVA", "Spring", 2024),
    ("Spring2024/Wellesley_Spring2024_Tab_Card.pdf", "Wellesley", "Spring", 2024),
    ("Spring2024/William&Mary_Spring2024_Tab_Card.pdf", "William & Mary", "Spring", 2024),
    ("Spring2024/Williams_Spring2024_Tab_Card.pdf", "Williams", "Spring", 2024),
    ("Spring2024/Yale_Spring2024_Tab_Card.pdf", "Yale", "Spring", 2024),
]

FALL_2023_TAB_CARDS = [
    ("Fall2023/Bates_Fall2023_Tab_Card.pdf", "Bates", "Fall", 2023),
    ("Fall2023/Binghamton_Fall2023_Tab_Card.pdf", "Binghamton", "Fall", 2023),
    ("Fall2023/Boston_Fall2023_Tab_Card.pdf", "Boston", "Fall", 2023),
    ("Fall2023/BrownAndWesleyan_Fall2023_Tab_Card.pdf", "Brown/Wesleyan", "Fall", 2023),
    ("Fall2023/Columbia_Fall2023_Tab_Card.pdf", "Columbia", "Fall", 2023),
    ("Fall2023/Drexel_Fall2023_Tab_Card.pdf", "Drexel", "Fall", 2023),
    ("Fall2023/Fordham_Fall2023_Tab_Card.pdf", "Fordham", "Fall", 2023),
    ("Fall2023/Georgetown_Fall2023_Tab_Card.pdf", "Georgetown", "Fall", 2023),
    ("Fall2023/GW_Fall2023_Tab_Card.pdf", "GW", "Fall", 2023),
    ("Fall2023/Northeastern_Fall2023_Tab_Card.pdf", "Northeastern", "Fall", 2023),
    ("Fall2023/UMD_Fall2023_Tab_Card.pdf", "UMD", "Fall", 2023),
    ("Fall2023/UVA_Fall2023_Tab_Card.pdf", "UVA", "Fall", 2023),
    ("Fall2023/Williams_Fall2023_Tab_Card.pdf", "Williams", "Fall", 2023),
]

SPRING_2023_TAB_CARDS = [
    ("Spring2023/Boston_Spring2023_Tab_Card.pdf", "Boston", "Spring", 2023),
    ("Spring2023/Brown_Spring2023_Tab_Card.pdf", "Brown", "Spring", 2023),
    ("Spring2023/Chicago_Spring2023_Tab_Card.pdf", "Chicago", "Spring", 2023),
    ("Spring2023/CMU_Spring2023_Tab_Card.pdf", "CMU", "Spring", 2023),
    ("Spring2023/Georgetown_Spring2023_Tab_Card.pdf", "Georgetown", "Spring", 2023),
    ("Spring2023/GW_Spring2023_Tab_Card.pdf", "GW", "Spring", 2023),
    ("Spring2023/Northeastern_Spring2023_Tab_Card.pdf", "Northeastern", "Spring", 2023),
    ("Spring2023/NYU_Spring2023_Tab_Card.pdf", "NYU", "Spring", 2023),
    ("Spring2023/Princeton_Spring2023_Tab_Card.pdf", "Princeton", "Spring", 2023),
    ("Spring2023/Swarthmore_Spring2023_Tab_Card.pdf", "Swarthmore", "Spring", 2023),
    ("Spring2023/TheCollegeOfNewJersey_Spring2023_Tab_Card.pdf", "The College of New Jersey", "Spring", 2023),
    ("Spring2023/UMD_Spring2023_Tab_Card.pdf", "UMD", "Spring", 2023),
    ("Spring2023/UVA_Spring2023_Tab_Card.pdf", "UVA", "Spring", 2023),
    ("Spring2023/Boston_Spring2023_Tab_Card.pdf", "Boston", "Spring", 2023),
    ("Spring2023/Wellesley_Spring2023_Tab_Card.pdf", "Wellesley", "Spring", 2023),
    ("Spring2023/Wesleyan_Spring2023_Tab_Card.pdf", "Wesleyan", "Spring", 2023),
    ("Spring2023/Yale_Spring2023_Tab_Card.pdf", "Yale", "Spring", 2023),
]

FALL_2022_TAB_CARDS = [
    ("Fall2022/Binghamton_Fall2022_Tab_Card.pdf", "Binghamton", "Fall", 2022),
    ("Fall2022/Brandeis_Fall2022_Tab_Card.pdf", "Brandeis", "Fall", 2022),
    ("Fall2022/Columbia_Fall2022_Tab_Card.pdf", "Columbia", "Fall", 2022),
    ("Fall2022/Fordham_Fall2022_Tab_Card.pdf", "Fordham", "Fall", 2022),
    ("Fall2022/Georgetown_Fall2022_Tab_Card.pdf", "Georgetown", "Fall", 2022),
    ("Fall2022/GW_Fall2022_Tab_Card.pdf", "GW", "Fall", 2022),
    ("Fall2022/Hopkins_Fall2022_Tab_Card.pdf", "Hopkins", "Fall", 2022),
    ("Fall2022/Northeastern_Fall2022_Tab_Card.pdf", "Northeastern", "Fall", 2022),
    ("Fall2022/Pitt_Fall2022_Tab_Card.pdf", "Pitt", "Fall", 2022),
    ("Fall2022/Smith_Fall2022_Tab_Card.pdf", "Smith", "Fall", 2022),
    ("Fall2022/Swarthmore_Fall2022_Tab_Card.pdf", "Swarthmore", "Fall", 2022),
    ("Fall2022/UMD_Fall2022_Tab_Card.pdf", "UMD", "Fall", 2022),
    ("Fall2022/UVA_Fall2022_Tab_Card.pdf", "UVA", "Fall", 2022),
    ("Fall2022/Wellesley_Fall2022_Tab_Card.pdf", "Wellesley", "Fall", 2022),
    ("Fall2022/WilliamAndMary_Fall2022_Tab_Card.pdf", "William & Mary", "Fall", 2022),
]

SPRING_2022_TAB_CARDS = [
    ("Spring2022/Boston_Spring2022_Tab_Card.pdf", "Boston", "Spring", 2022),
    ("Spring2022/Brandeis_Spring2022_Tab_Card.pdf", "Brandeis", "Spring", 2022),
    ("Spring2022/Brown_Spring2022_Tab_Card.pdf", "Brown", "Spring", 2022),
    ("Spring2022/Georgetown_Spring2022_Tab_Card.pdf", "Georgetown", "Spring", 2022),
    ("Spring2022/GW_Spring2022_Tab_Card.pdf", "GW", "Spring", 2022),
    ("Spring2022/Northeastern_Spring2022_Tab_Card.pdf", "Northeastern", "Spring", 2022),
    ("Spring2022/NYU_Spring2022_Tab_Card.pdf", "NYU", "Spring", 2022),
    ("Spring2022/PittCMU_Spring2022_Tab_Card.pdf", "Pitt/CMU", "Spring", 2022),
    ("Spring2022/Smith_Spring2022_Tab_Card.pdf", "Smith", "Spring", 2022),
    ("Spring2022/Swarthmore_Spring2022_Tab_Card.pdf", "Swarthmore", "Spring", 2022),
    ("Spring2022/UMD_Spring2022_Tab_Card.pdf", "UMD", "Spring", 2022),
    ("Spring2022/UVA_Spring2022_Tab_Card.pdf", "UVA", "Spring", 2022),
    ("Spring2022/WashU_Spring2022_Tab_Card.pdf", "WashU", "Spring", 2022),
    ("Spring2022/Wellesley_Spring2022_Tab_Card.pdf", "Wellesley", "Spring", 2022),
    ("Spring2022/Wesleyan_Spring2022_Tab_Card.pdf", "Wesleyan", "Spring", 2022),
    ("Spring2022/WilliamAndMary_Spring2022_Tab_Card.pdf", "William & Mary", "Spring", 2022),
]

FALL_2021_TAB_CARDS = [
    ("Fall2021/Binghamton_Fall2021_Tab_Card.pdf", "Binghamton", "Fall", 2021),
    ("Fall2021/JHUAU_Fall2021_Tab_Card.pdf", "John's Hopkins/American", "Fall", 2021),  
    ("Fall2021/NUBC_Fall2021_Tab_Card.pdf", "Northeastern/Boston", "Fall", 2021),
    ("Fall2021/SmithColumbia_Fall2021_Tab_Card.pdf", "Smith/Columbia", "Fall", 2021),
    ("Fall2021/TempleWesleyan_Fall2021_Tab_Card.pdf", "Temple/Wesleyan", "Fall", 2021),
    ("Fall2021/WilliamAndMary_Fall2021_Tab_Card.pdf", "Willian & Mary", "Fall", 2021),
]

SPRING_2021_TAB_CARDS = [
    ("Spring2021/Amherst_Spring2021_Tab_Card.pdf", "Amherst", "Spring", 2021),
    ("Spring2021/ChicagoNortheastern_Spring2021_Tab_Card.pdf", "Chicago/Northeastern", "Spring", 2021),
    ("Spring2021/CMU_Spring2021_Tab_Card.pdf", "CMU", "Spring", 2021),
    ("Spring2021/Dartmouth_Spring2021_Tab_Card.pdf", "Dartmouth", "Spring", 2021),
    ("Spring2021/GW_Spring2021_Tab_Card.pdf", "GW", "Spring", 2021),
    ("Spring2021/NYUWashU_Spring2021_Tab_Card.pdf", "NYU/WashU", "Spring", 2021),
    ("Spring2021/PrincetonNUBates_Spring2021_Tab_Card.pdf", "Princeton/Northeastern/Bates", "Spring", 2021),
    ("Spring2021/UMDBU_Spring2021_Tab_Card.pdf", "UMD/Boston", "Spring", 2021),
    ("Spring2021/WesleyanTCNJ_Spring2021_Tab_Card.pdf", "Wesleyan/The College of New Jersey", "Spring", 2021),
]

FALL_2020_TAB_CARDS = [
    ("Fall2020/UVAWDS_Fall2020_Tab_Card.pdf", "UVA/Wellesley", "Fall", 2020),
    ("Fall2020/CUNYUMD_Fall2020_Tab_Card.pdf", "CUNY/UMD", "Fall", 2020),
    ("Fall2020/Columbia_Fall2020_Tab_Card.pdf", "Columbia", "Fall", 2020),
]

SPRING_2020_TAB_CARDS = [
    ("Spring2020/CMU_Spring2020_Tab_Card.pdf", "CMU", "Spring", 2020),
    ("Spring2020/ColumbiaSwarthmore_Spring2020_Tab_Card.pdf", "Columbia/Swarthmore", "Spring", 2020),
    ("Spring2020/CUNY_Spring2020_Tab_Card.pdf", "CUNY", "Spring", 2020),
    ("Spring2020/GW_Spring2020_Tab_Card.pdf", "GW", "Spring", 2020),
    ("Spring2020/Rutgers_Spring2020_Tab_Card.pdf", "Rutgers", "Spring", 2020),
    ("Spring2020/Swarthmore_Spring2020_Tab_Card.pdf", "Swarthmore", "Spring", 2020),
    ("Spring2020/Tufts_Spring2020_Tab_Card.pdf", "Tufts", "Spring", 2020),
    ("Spring2020/Tufts2_Spring2020_Tab_Card.pdf", "Tufts", "Spring", 2020),
    ("Spring2020/UVA_Spring2020_Tab_Card.pdf", "UVA", "Spring", 2020),
    ("Spring2020/Yale_Spring2020_Tab_Card.pdf", "Yale", "Spring", 2020),
]

FALL_2019_TAB_CARDS = [
    ("Fall2019/American_Fall2019_Tab_Card.pdf", "American", "Fall", 2019),
    ("Fall2019/Amherst_Fall2019_Tab_Card.pdf", "Amherst", "Fall", 2019),
    ("Fall2019/Binghamton_Fall2019_Tab_Card.pdf", "Binghamton", "Fall", 2019),
    ("Fall2019/Brown_Fall2019_Tab_Card.pdf", "Brown", "Fall", 2019),
    ("Fall2019/CUNY_Fall2019_Tab_Card.pdf", "CUNY", "Fall", 2019),
    ("Fall2019/GW_Fall2019_Tab_Card.pdf", "GW", "Fall", 2019),
    ("Fall2019/Hopkins_Fall2019_Tab_Card.pdf", "Hopkins", "Fall", 2019),
    ("Fall2019/Penn_Fall2019_Tab_Card.pdf", "Penn", "Fall", 2019),
    ("Fall2019/Pitt_Fall2019_Tab_Card.pdf", "Pitt", "Fall", 2019),
    ("Fall2019/Smith_Fall2019_Tab_Card.pdf", "Smith", "Fall", 2019),
    ("Fall2019/Swarthmore_Fall2019_Tab_Card.pdf", "Swarthmore", "Fall", 2019),
    ("Fall2019/TCNJ_Fall2019_Tab_Card.pdf", "The College of New Jersey", "Fall", 2019),
    ("Fall2019/UMD_Fall2019_Tab_Card.pdf", "UMD", "Fall", 2019),
    ("Fall2019/Villanova_Fall2019_Tab_Card.pdf", "Villanova", "Fall", 2019),
    ("Fall2019/Wellesley_Fall2019_Tab_Card.pdf", "Wellesley", "Fall", 2019),
]

SPRING_2019_TAB_CARDS = [
    ("Spring2019/American_Spring2019_Tab_Card.pdf", "American", "Spring", 2019),
    ("Spring2019/Brown_Spring2019_Tab_Card.pdf", "Brown", "Spring", 2019),
    ("Spring2019/CMU_Spring2019_Tab_Card.pdf", "CMU", "Spring", 2019),
    ("Spring2019/Dartmouth_Spring2019_Tab_Card.pdf", "Dartmouth", "Spring", 2019),
    ("Spring2019/FranklinAndMarshall_Spring2019_Tab_Card.pdf", "Franklin and Marshall", "Spring", 2019),
    ("Spring2019/GW_Spring2019_Tab_Card.pdf", "GW", "Spring", 2019),
    ("Spring2019/NYU_Spring2019_Tab_Card.pdf", "NYU", "Spring", 2019),
    ("Spring2019/Stanford_Spring2019_Tab_Card.pdf", "Stanford", "Spring", 2019),
    ("Spring2019/Swarthmore_Spring2019_Tab_Card.pdf", "Swarthmore", "Spring", 2019),
    ("Spring2019/TCNJ_Spring2019_Tab_Card.pdf", "The College of New Jersey", "Spring", 2019),
    ("Spring2019/Tufts_Spring2019_Tab_Card.pdf", "Tufts", "Spring", 2019),
    ("Spring2019/UVA_Spring2019_Tab_Card.pdf", "UVA", "Spring", 2019),
    ("Spring2019/Villanova_Spring2019_Tab_Card.pdf", "Villanova", "Spring", 2019),
    ("Spring2019/WestPoint_Spring2019_Tab_Card.pdf", "West Point", "Spring", 2019),
]

Several tournaments were excluded for bad formating, lack of tab cards, or lack of permission to the documnets. The following describes those ommited:

In [794]:
from tabulate import tabulate
omitted_tables = [["Spring2019","Wellesley, Princeton","",""],["Fall2019","Fordham, UVA, Columbia","",""],
                  ["Spring2020","","GU, Williams",""],["Fall2020","","Harvard, Swarthmore",""],
                  ["Spring2021","Swarthmore","Brandeis, W&M, GU, Yale",""],["Fall2021","","","Tufts/UMD, GW/Fordham, Brown, Harvard/Penn"],
                  ["Spring2022","Princeton, Yale, Rutgers","Brown, Williams","Darthmouth"],["Fall2022","","Brown, Yale, Williams","Tufts, Harvard, Rutgers"],
                  ["Spring2023","Hopkins","Brandeis, Rutgers, Penn, Williams, UChicago","UMass"],["Fall2023","Hopkins, Swarthmore","Brandeis, Harvard",""],
                  ["Spring2024","Amherst","UMass, Brandeis, NYU, Windsor, Tufts",""],["Fall2024","","Harvard, Binghamton, Tufts, Brown",""],
                  ["Spring2025","Rutgers, Amherst, Penn, UVA","","Dartmouth, UT Austin"],["Fall2025","Drexel","","Brandeis, Bates, American"],
                  ["Spring2026","Temple","",""]]

print(tabulate(omitted_tables, headers=["Season","Bad Format", "Lacking Permission", "Missing Tab Card"], tablefmt="grid"))

+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Season     | Bad Format                  | Lacking Permission                          | Missing Tab Card                           |
+============+=============================+=============================================+============================================+
| Spring2019 | Wellesley, Princeton        |                                             |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Fall2019   | Fordham, UVA, Columbia      |                                             |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Spring2020 |                             | GU,

Below are the regex patterns used for normalization and sanitization as well as the functions used to parse and clean the df:

In [780]:
# pre-compile regex patterns
NON_ASCII_RX = re.compile(r"[^\x20-\x7E]")
MULTISPACE_RX = re.compile(r"\s+")
STATUS_PAREN_RX = re.compile(r"\s*\(([NV])\)\s*$")
CLEAN_TRAILING_RX = re.compile(r"\s+$")
TEAM_PREFIX_RX = re.compile(r"Team:\s*(.+)")

In [479]:
def normalize_team_name(name):
    if name is None:
        return ""
    stripped = NON_ASCII_RX.sub("", str(name))
    return MULTISPACE_RX.sub(" ", stripped).strip()
    
def clean(val):
    if val is None:
        return ""
    return MULTISPACE_RX.sub(" ", str(val)).strip()

def remove_status(name):
    return STATUS_TAG_RX.sub(r"\1", name)

def split_name_and_status(raw_name):
    if not raw_name:
        return "", None
    name = normalize_team_name(raw_name)
    match = STATUS_PAREN_RX.search(name)
    if match:
        status = "Novice" if match.group(1) == "N" else "Varsity"
        clean_name = STATUS_PAREN_RX.sub("", name).strip()
        return clean_name, status    
    return name, None
    
def parse_tab_cards(card):
    all_rows, speaker_lookup = [], []
    current_team, current_speakers, pending_label = None, None, None
    
    with pdfplumber.open(card) as pdf:
        for page in pdf.pages:
            team_labels = []
            for line in page.extract_text_lines():
                match = TEAM_PREFIX_RX.search(line["text"])
                if match:
                    team_labels.append({"name": match.group(1).strip(), "top": line["top"]})
 
            tables = page.find_tables()
 
            for table in tables:
                data = table.extract()
                if not data or len(data) < 1:
                    continue
    
                first_cell = clean(data[0][0]) if data[0] else ""
                has_header = (first_cell in ("R", ""))
 
                if has_header:
                    header = [clean(c) for c in data[0]]
                    body = data[1:]
                    speaker1_header = header[5] if len(header) > 5 else ""
                    speaker2_header = header[6] if len(header) > 6 else ""
                else:
                    body = data
                    speaker1_header, speaker2_header = current_speakers if current_speakers else ("", "")
 
                table_top = table.bbox[1]
                labels_above = [t for t in team_labels if t["top"] <= table_top]
 
                if labels_above:
                    team_name = max(labels_above, key=lambda t: t["top"])["name"]
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                elif has_header and current_speakers and (speaker1_header, speaker2_header) == current_speakers:
                    team_name = current_team
                elif not has_header and current_team is not None:
                    team_name = current_team
                elif pending_label is not None:
                    team_name = pending_label
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                else:
                    team_name = current_team
                    current_speakers = (speaker1_header, speaker2_header)
 
                if not any(s["Team"] == team_name for s in speaker_lookup):
                    name1, status1 = split_name_and_status(speaker1_header)
                    name2, status2 = split_name_and_status(speaker2_header)
                    speaker_lookup.append({
                        "Team": team_name,
                        "Speaker One Name": name1, "Speaker One Status": status1,
                        "Speaker Two Name": name2, "Speaker Two Status": status2,
                    })
 
                for row in body:
                    row = [clean(c) for c in row]
                    if not row or row[0].lower().startswith("tournament totals"):
                        continue  
 
                    row = (row + [""] * 8)[:8]
                    round_no, g_o, w_l, opponent, judge, sp1, sp2, total = row
 
                    if not round_no:
                        if not any([g_o, w_l, opponent, judge, sp1, sp2]):
                            continue
                        round_no = "UNKNOWN (split across page break)"
 
                    all_rows.append({
                        "Team": team_name, "Round": round_no, "G/O": g_o, "W/L": w_l,
                        "Opponent": opponent, "Judge": judge, 
                        "Speaker One Score": sp1, "Speaker Two Score": sp2, "Total": total,
                    })
 
            if team_labels:
                last_label = max(team_labels, key=lambda t: t["top"])
                table_tops = [t.bbox[1] for t in tables]
                if not any(last_label["top"] <= top for top in table_tops):
                    pending_label = last_label["name"]
 
    return pd.DataFrame(all_rows), pd.DataFrame(speaker_lookup)

def split_speaks_and_ranks(df, score_col, prefix):
    if score_col not in df.columns:
        return df

    cleaned_series = df[score_col].astype(str).str.replace(r"[\(\)\s]", "", regex=True)
    split_data = cleaned_series.str.split(",", expand=True)

    if split_data.shape[1] < 2:
        split_data = pd.DataFrame(index=df.index, columns=[0, 1])

    df[f"{prefix} Speaks"] = pd.to_numeric(split_data[0], errors="coerce")
    df[f"{prefix} Rank"] = pd.to_numeric(split_data[1], errors="coerce").astype("Int64") # Capital I allows NaN integers

    return df.drop(columns=[score_col])

def add_speaker_and_opponent_names(all_df, speaker_df):
    speaker_df = speaker_df.copy()
    speaker_df["_key"] = speaker_df["Team"].map(normalize_team_name)
    speaker_df = speaker_df.drop_duplicates(subset="_key", keep="first")
    
    lookup = speaker_df.set_index("_key")[
        ["Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status"]
    ]
 
    df = all_df.copy()
    df["_team_key"] = df["Team"].map(normalize_team_name)
    df["_opponent_key"] = df["Opponent"].map(normalize_team_name)
 
    df = df.merge(lookup, left_on="_team_key", right_index=True, how="left")
    
    df = df.merge(
        lookup.rename(columns={
            "Speaker One Name": "Opponent One Name", "Speaker One Status": "Opponent One Status",
            "Speaker Two Name": "Opponent Two Name", "Speaker Two Status": "Opponent Two Status"
        }),
        left_on="_opponent_key", right_index=True, how="left",
    )
 
    score_lookup = all_df.copy()
    score_lookup["_opponent_key"] = score_lookup["Team"].map(normalize_team_name)
    score_lookup = score_lookup[["_opponent_key", "Round", "Speaker One Score", "Speaker Two Score"]].rename(
        columns={"Speaker One Score": "Opponent Speaker One Score", "Speaker Two Score": "Opponent Speaker Two Score"}
    )
    df = df.merge(score_lookup, on=["_opponent_key", "Round"], how="left")
    df = df.drop(columns=["Team", "Opponent", "_team_key", "_opponent_key"])

    df = split_speaks_and_ranks(df, "Speaker One Score", "Speaker One")
    df = split_speaks_and_ranks(df, "Speaker Two Score", "Speaker Two")
    df = split_speaks_and_ranks(df, "Opponent Speaker One Score", "Opponent One")
    df = split_speaks_and_ranks(df, "Opponent Speaker Two Score", "Opponent Two")
    
    ordered_cols = [c for c in [
        "Tournament", "Season", "Year", "Round", "G/O", "W/L",
        "Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status",
        "Opponent One Name", "Opponent One Status", "Opponent Two Name", "Opponent Two Status",
        "Judge", "Speaker One Speaks", "Speaker One Rank", "Speaker Two Speaks", "Speaker Two Rank",
        "Opponent One Speaks", "Opponent One Rank", "Opponent Two Speaks", "Opponent Two Rank", "Total",
    ] if c in df.columns]
    
    return df[ordered_cols]

def process_all_cards(tab_cards):
    per_tournament_results = []
    all_speaker_rows = [] 
 
    for pdf_path, tournament_name, season, year in tab_cards:
        all_df, speaker_df = parse_tab_cards(pdf_path)
        if all_df.empty:
            continue
            
        all_df = all_df.drop(columns=["Total"], errors="ignore")
        all_df.insert(0, "Tournament", tournament_name)
        all_df.insert(1, "Season", season)
        all_df.insert(2, "Year", year)
 
        named_df = add_speaker_and_opponent_names(all_df, speaker_df)
        per_tournament_results.append(named_df)
 
        speaker_df = speaker_df.copy()
        speaker_df.insert(0, "Tournament", tournament_name)
        speaker_df.insert(1, "Season", season)
        speaker_df.insert(2, "Year", year)
        all_speaker_rows.append(speaker_df)
 
    return pd.concat(per_tournament_results, ignore_index=True), pd.concat(all_speaker_rows, ignore_index=True)

def collapse_to_one_row_per_debate(combined_df):
    df = combined_df.copy()
    
    gov_rows = df[df["G/O"] == "G"].copy()
    opp_rows = df[df["G/O"] == "O"].copy()
    other_rows = df[~df["G/O"].isin(["G", "O"])].copy() # BYEs, drops, etc.

    def get_speaker_keys(df_slice, sp1_col, sp2_col):
        s1 = df_slice[sp1_col].fillna("").astype(str)
        s2 = df_slice[sp2_col].fillna("").astype(str)
        return df_slice["Tournament"] + "||" + df_slice["Round"] + "||" + \
               pd.concat([s1, s2], axis=1).min(axis=1) + "||" + pd.concat([s1, s2], axis=1).max(axis=1)

    gov_rows["_match_key"] = get_speaker_keys(gov_rows, "Opponent One Name", "Opponent Two Name")
    opp_rows["_match_key"] = get_speaker_keys(opp_rows, "Speaker One Name", "Speaker Two Name")

    merged_df = pd.merge(
        gov_rows, 
        opp_rows, 
        on="_match_key", 
        how="outer", 
        suffixes=("_gov", "_opp")
    )

    result_df = pd.DataFrame()
    
    result_df["Tournament"] = merged_df["Tournament_gov"].fillna(merged_df["Tournament_opp"])
    result_df["Season"] = merged_df["Season_gov"].fillna(merged_df["Season_opp"])
    result_df["Year"] = merged_df["Year_gov"].fillna(merged_df["Year_opp"])
    result_df["Round"] = merged_df["Round_gov"].fillna(merged_df["Round_opp"])
    
    result_df["Gov Speaker One Name"] = merged_df["Speaker One Name_gov"]
    result_df["Gov Speaker One Status"] = merged_df["Speaker One Status_gov"]
    result_df["Gov Speaker One Speaks"] = merged_df["Speaker One Speaks_gov"]
    result_df["Gov Speaker One Rank"] = merged_df["Speaker One Rank_gov"]
    
    result_df["Gov Speaker Two Name"] = merged_df["Speaker Two Name_gov"]
    result_df["Gov Speaker Two Status"] = merged_df["Speaker Two Status_gov"]
    result_df["Gov Speaker Two Speaks"] = merged_df["Speaker Two Speaks_gov"]
    result_df["Gov Speaker Two Rank"] = merged_df["Speaker Two Rank_gov"]
    
    result_df["Opp Speaker One Name"] = merged_df["Speaker One Name_opp"]
    result_df["Opp Speaker One Status"] = merged_df["Speaker One Status_opp"]
    result_df["Opp One Speaks"] = merged_df["Speaker One Speaks_opp"]
    result_df["Opp One Rank"] = merged_df["Speaker One Rank_opp"]
    
    result_df["Opp Speaker Two Name"] = merged_df["Speaker Two Name_opp"]
    result_df["Opp Speaker Two Status"] = merged_df["Speaker Two Status_opp"]
    result_df["Opp Two Speaks"] = merged_df["Speaker Two Speaks_opp"]
    result_df["Opp Two Rank"] = merged_df["Speaker Two Rank_opp"]
    
    result_df["Winner"] = None
    result_df.loc[merged_df["W/L_gov"] == "W", "Winner"] = "Gov"
    result_df.loc[merged_df["W/L_gov"] == "L", "Winner"] = "Opp"
    result_df.loc[(merged_df["W/L_gov"].isna()) & (merged_df["W/L_opp"] == "W"), "Winner"] = "Opp"
    result_df.loc[(merged_df["W/L_gov"].isna()) & (merged_df["W/L_opp"] == "L"), "Winner"] = "Gov"
    
    result_df["Judge"] = merged_df["Judge_gov"].fillna(merged_df["Judge_opp"])

    if not other_rows.empty:
        other_clean = pd.DataFrame({
            "Tournament": other_rows["Tournament"], "Season": other_rows["Season"], "Year": other_rows["Year"], "Round": other_rows["Round"],
            "Gov Speaker One Name": other_rows["Speaker One Name"], "Gov Speaker One Status": other_rows["Speaker One Status"], "Gov Speaker One Speaks": other_rows["Speaker One Speaks"], "Gov Speaker One Rank": other_rows["Speaker One Rank"],
            "Gov Speaker Two Name": other_rows["Speaker Two Name"], "Gov Speaker Two Status": other_rows["Speaker Two Status"], "Gov Speaker Two Speaks": other_rows["Speaker Two Speaks"], "Gov Speaker One Rank": other_rows["Speaker One Rank"],
            "Opp Speaker One Name": None, "Opp Speaker One Status": None, "Opp Speaker One Speak": None, "Opp Speaker One Rank": None,
            "Opp Speaker Two Name": None, "Opp Speaker Two Status": None, "Opp Speaker Two Score": None, "Opp Speaker Two Rank": None,
            "Winner": other_rows["G/O"].apply(lambda x: "BYE" if x == "BYE" else None),
            "Judge": other_rows["Judge"]
        })
        result_df = pd.concat([result_df, other_clean], ignore_index=True)

    return result_df

...

In [480]:
semester_variables = [(SPRING_2026_TAB_CARDS, "Spring_2026.csv"), (FALL_2025_TAB_CARDS, "Fall_2025.csv"), 
                      (SPRING_2025_TAB_CARDS, "Spring_2025.csv"), (FALL_2024_TAB_CARDS, "Fall_2024.csv"),
                      (SPRING_2024_TAB_CARDS, "Spring_2024.csv"), (FALL_2023_TAB_CARDS, "Fall_2023.csv"), 
                      (SPRING_2023_TAB_CARDS, "Spring_2023.csv"), (FALL_2022_TAB_CARDS, "Fall_2022.csv"), 
                      (SPRING_2022_TAB_CARDS, "Spring_2022.csv"), (FALL_2021_TAB_CARDS, "Fall_2021.csv"),
                      (SPRING_2021_TAB_CARDS, "Spring_2021.csv"), (FALL_2020_TAB_CARDS, "Fall_2020.csv"),
                      (SPRING_2020_TAB_CARDS, "Spring_2020.csv"), (FALL_2019_TAB_CARDS, "Fall_2019.csv"),
                      (SPRING_2019_TAB_CARDS, "Spring_2019.csv")]
dfs = []
speakers = []
for (t, label) in semester_variables:
    df, speaker = process_all_cards(t)
    df = df[df["G/O"].fillna("").str.strip() != ""]
    df = df[((df["G/O"] != "BYE") & (df["Opponent One Name"].fillna("").str.strip() != "")) | (df["G/O"] == "BYE")]
    df.to_csv(label, index=False)
    dfs.append(df)
    speakers.append(speaker)

In [586]:
rounds_over_time = pd.concat(dfs, ignore_index=True)
rounds_over_time = pd.concat([Spring_2026_combined_df, Fall_2025_combined_df, Spring_2025_combined_df, Fall_2024_combined_df, Spring_2024_combined_df, Fall_2023_combined_df, Spring_2023_combined_df, Fall_2022_combined_df, Spring_2022_combined_df, Fall_2021_combined_df, Spring_2021_combined_df, Fall_2020_combined_df, Spring_2020_combined_df, Fall_2019_combined_df, Spring_2019_combined_df], ignore_index=True)
rounds_over_time = rounds_over_time[rounds_over_time["G/O"]!= ""]
rounds_over_time = rounds_over_time[((rounds_over_time["G/O"] != "BYE") & (rounds_over_time["Opponent One Name"].fillna("").str.strip() != "")) | (rounds_over_time["G/O"] == "BYE")]
rounds_over_time = rounds_over_time[rounds_over_time["Speaker One Name"].fillna("").str.strip() != ""]
rounds_over_time['Judge'] = (
    rounds_over_time['Judge']
    .str.split(' - ').str[0]
    .str.replace(r'\s*\(V\)\s*', '', regex=True)
    .str.strip()
)

rounds_over_time.to_csv("Dirty_All_Rounds.csv", index=False)

In [754]:
clean_rounds_over_time = pd.read_csv("Manual_All_Rounds.csv")

df_one = clean_rounds_over_time[["Speaker One Name"]].rename(columns={"Speaker One Name": "Debater"})
df_two = clean_rounds_over_time[["Speaker Two Name"]].rename(columns={"Speaker Two Name": "Debater"})
df_judge = clean_rounds_over_time[["Judge"]].rename(columns={"Judge": "Debater"})
df_debaters = pd.concat([df_one, df_two, df_judge], ignore_index=True)
df_debaters["Debater"] = df_debaters["Debater"].str.upper()
df_debaters = df_debaters[df_debaters["Debater"].fillna("").str.strip() != ""]
df_debaters = df_debaters.drop_duplicates()

In [756]:
valid_debaters = pd.read_csv("test.csv")
sorted_valid_df = pd.merge(df_debaters, valid_debaters, on='Debater', how='inner')
sorted_valid_df = sorted_valid_df.drop_duplicates()
merged_df.to_csv("All_Debaters.csv", index=False)

In [757]:
df_judges = df_judge[df_judge["Debater"].fillna("").str.strip() != ""]
df_judges = df_judges.drop_duplicates()
judge_result = (df_judge.merge(sorted_valid_df, how="left", indicator=True) .query('_merge == "left_only"') .drop(columns="_merge") )
judge_result.to_csv("Judges.csv", index=False)

In [758]:
valid_name_rows = set(sorted_valid_df["Debater"])
clean_rounds_over_time = clean_rounds_over_time[
    #clean_rounds_over_time["Speaker One Name"].isin(valid_names)
    #& clean_rounds_over_time["Speaker Two Name"].isin(valid_names)
    clean_rounds_over_time["Judge"].isin(valid_names)
]
clean_rounds_over_time.to_csv("All_Rounds.csv", index=False)

## Exploratory Analysis

In [760]:
clean_rounds_over_time = clean_rounds_over_time.rename(columns={"Judge": "Debater"})
judge_school_df = pd.merge(clean_rounds_over_time, merged_df, on='Debater', how='inner')
judge_school_df = judge_school_df.rename(columns={"Debater": "Judge", "University": "Judge School", "Starting Year": "Judge Start"})
judge_school_df = judge_school_df.drop_duplicates()

judge_school_df.to_csv("Rounds_With_Judge_School.csv", index=False)

In [762]:
speaks_one = judge_school_df.groupby('Judge School')['Speaker One Speaks'].mean()
speaks_two = judge_school_df.groupby('Judge School')['Speaker Two Speaks'].mean()
opp_speaks_one = judge_school_df.groupby('Judge School')['Opponent One Speaks'].mean()
opp_speaks_two = judge_school_df.groupby('Judge School')['Opponent Two Speaks'].mean()
count = judge_school_df.groupby('Judge School')['Opponent Two Speaks'].count()
total_speaks_avg = (speaks_one + speaks_two + opp_speaks_one + opp_speaks_two) / 4

### Speaks

## Insights and Conclusions